# Python Week-2
## 建立CoLabNotebook
## 執行程式


In [ ]:
course_name = "Python與資料科學"

print("歡迎來到:",course_name)

歡迎來到: Python與資料科學


In [ ]:
a=10
b=11
print(a+b)

21


In [ ]:
stray= 'hello world~~'
print(stray)

hello world~~


In [ ]:
# ============================================================
#  Google Colab 實用功能示範
#  執行方式：複製整段 -> 貼到 Colab 新儲存格 -> Shift+Enter
# ============================================================

# ------------------------------------------------------------
# 1️⃣ 環境檢查：確認是否有 GPU 可用
# ------------------------------------------------------------
import torch, tensorflow as tf, sys, subprocess, json, os, textwrap, math, random, datetime

print("🐍 Python 版本 :", sys.version.split()[0])
print("🔧 PyTorch 版本 :", torch.__version__)
print("🔧 TensorFlow 版本 :", tf.__version__)

# 檢查 GPU
gpu_info = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("✅ 偵測到 GPU：")
    for line in gpu_info.stdout.strip().split("\n"):
        print("   -", line)
    # PyTorch / TF 也確認一下
    print("   PyTorch cuda 可用 :", torch.cuda.is_available())
    print("   TF GPU 裝置      :", tf.config.list_physical_devices('GPU'))
else:
    print("⚠️ 未偵測到 GPU（執行階段 → 變更執行階段類型 → 選 GPU 可啟用）")

print("\n" + "="*60)

# ------------------------------------------------------------
# 2️⃣ 視覺化：畫一個漂亮的數學函數動畫圖
# ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

x = np.linspace(-2*np.pi, 2*np.pi, 400)
y1 = np.sin(x)
y2 = np.cos(x)
y3 = np.sin(x) * np.exp(-0.1*np.abs(x))  # 振幅遞減的正弦波

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_title("Math Functions Visualization", fontsize=14)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(True, alpha=0.3)
ax.set_xlim(-2*np.pi, 2*np.pi)
ax.set_ylim(-1.5, 1.5)

line1, = ax.plot([], [], label="sin(x)", color="#1f77b4", linewidth=2)
line2, = ax.plot([], [], label="cos(x)", color="#ff7f0e", linewidth=2, linestyle="--")
line3, = ax.plot([], [], label="sin(x)·e^(-0.1|x|)", color="#2ca02c", linewidth=2, linestyle=":")
ax.legend(loc="upper right")

def init():
    for line in (line1, line2, line3):
        line.set_data([], [])
    return line1, line2, line3

def animate(i):
    # 逐步繪製
    idx = min(i*5, len(x))
    line1.set_data(x[:idx], y1[:idx])
    line2.set_data(x[:idx], y2[:idx])
    line3.set_data(x[:idx], y3[:idx])
    return line1, line2, line3

ani = animation.FuncAnimation(fig, animate, frames=80, init_func=init,
                              interval=50, blit=True, repeat=False)
plt.close()  # 避免靜態圖重複顯示
display(HTML(ani.to_jshtml()))  # 在 Colab 直接播放動畫

print("="*60)

# ------------------------------------------------------------
# 3️⃣ 真實資料：從 UCI 下載「紅酒品質」資料集並做 EDA
# ------------------------------------------------------------
import pandas as pd
import seaborn as sns

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")

print("📊 資料集形狀 :", df.shape)
print("\n📋 前 5 筆 :")
display(df.head())

print("\n📈 基本統計 :")
display(df.describe().T.style.background_gradient(cmap="Blues"))

# 相關係數熱力圖
plt.figure(figsize=(10, 8))
arr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(arr, dtype=bool))
sns.heatmap(arr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap (Red Wine Quality)", fontsize=14)
plt.tight_layout()
plt.show()

print("="*60)

# ------------------------------------------------------------
# 4️⃣ 快速建模：Random Forest 預測酒品質（分類）
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# 將 quality 轉為三類：低(3-5)、中(6)、高(7-8)
def quality_to_class(q):
    if q <= 5: return 0  # Low
    elif q == 6: return 1  # Medium
    else: return 2  # High

df["quality_class"] = df["quality"].apply(quality_to_class)

X = df.drop(["quality", "quality_class"], axis=1)
y = df["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, max_depth=12,
                             random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("🎯 分類報告 :")
print(classification_report(y_test, y_pred,
                            target_names=["Low (≤5)", "Medium (6)", "High (≥7)"]))

# 混淆矩陣
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
                                        display_labels=["Low", "Medium", "High"],
                                        cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

# 特徵重要性
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(6, 5))
importances.plot(kind="barh", color="#1f77b4", edgecolor="black")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\n✅ 所有示範完成！你可以繼續在下方儲存格撰寫自己的程式碼。")

SyntaxError: positional argument follows keyword argument (127184206.py, line 52)

##Mission 1：執行第一段 Python 程式
#Knowledge Point
Notebook 會把程式分成多個儲存格。程式通常由上往下執行。
print() 用來把結果顯示出來。
文字需要用引號包起來，例如 "Hello"。
變數可以先儲存資訊，再放入 print() 中。

執行前預測：
請先猜測 print("歡迎來到", course_name) 會顯示什麼？

Try It：
請把 course_name 改成你自己的名字、學號或課程名稱，然後重新執行。

Verification：
檢查輸出是否變成你修改後的文字。

Explain-back：
請用自己的話說明：
1. course_name 裡面儲存了什麼？
2. print() 做了什麼事？
3. 為什麼修改 course_name 後，輸出也會改變？



In [4]:
# 建立一個變數，儲存課程名稱
course_name = "Python與資料科學"

# 使用 print() 顯示文字
print("歡迎來到", course_name)


歡迎來到 Python與資料科學


In [5]:
# 建立一個變數，儲存課程名稱
course_name = "Python與資料科學"

# 使用 print() 顯示文字
print("歡迎來到", course_name)


歡迎來到 Python與資料科學


##Mission 2：變數與資料型態
#Knowledge Point
* 變數是一個有名字的儲存位置。
= 在 Python 中表示「指定」或「存入」，
* 不是數學裡的相等判斷。
常見資料型態：

*   int：整數
*   float：浮點數
*   str：字串
*   bool：布林值，只有 True 或 False


* type() 可以用來檢查變數的資料型態。



執行前預測：
請猜測以下變數的 type() 輸出：
- student_count = 60
- average_sleep_hours = 6.7
- course_name = "Python與資料科學"
- has_missing_value = True

Try It：
把 average_sleep_hours 改成 7.3，再執行一次，觀察 type() 是否改變。不會

Verification：
確認 student_count 是 int，average_sleep_hours 是 float，
course_name 是 str，has_missing_value 是 bool。

Explain-back：
請說明：
1. 為什麼 = 在這裡不是數學中的「相等」？   因為它是指定
2. 6.7 和 6 的資料型態有什么不同？     6.7為浮點數，6為整數
3. True 是什麼資料型態？           布林值



In [2]:
# 建立不同資料型態的變數
student_count = 60
average_sleep_hours = 7.3
course_name = "Python與資料科學"
has_missing_value = True

# 使用 type() 檢查資料型態
print("student_count 的值：", student_count)
print("student_count 的型態：", type(student_count))

print("average_sleep_hours 的值：", average_sleep_hours)
print("average_sleep_hours 的型態：", type(average_sleep_hours))

print("course_name 的值：", course_name)
print("course_name 的型態：", type(course_name))

print("has_missing_value 的值：", has_missing_value)
print("has_missing_value 的型態：", type(has_missing_value))


student_count 的值： 60
student_count 的型態： <class 'int'>
average_sleep_hours 的值： 7.3
average_sleep_hours 的型態： <class 'float'>
course_name 的值： Python與資料科學
course_name 的型態： <class 'str'>
has_missing_value 的值： True
has_missing_value 的型態： <class 'bool'>


In [6]:
# 修改平均睡眠時數
average_sleep_hours = 7.3

# 檢查修改後的值與型態
print("修改後平均睡眠時數：", average_sleep_hours)
print("修改後型態：", type(average_sleep_hours))


修改後平均睡眠時數： 7.3
修改後型態： <class 'float'>


##Mission 3：串列、字典與索引
#Knowledge Point


*   串列 list 可以依順序存放多個資料，例如 ["A", "B", "C"]。
*   Python 的索引從 0 開始。
*   第一個元素是 0，第二個是 1，第三個是 2。
*   len() 可以用來計算串列中有幾個元素。
*   字典 dict 用「鍵」和「值」來儲存資料。
*   可以用字典的鍵來取得對應的值。



執行前預測：
departments[0] 會顯示哪個科系？
len(departments) 會顯示幾？

Try It：
把 sample_student["quiz_score"] 改成 76，再執行一次。

Verification：
確認 departments[0] 是第一筆科系。
確認 sample_student["department"] 能取得科系。
確認 sample_student["quiz_score"] 能取得成績。

Explain-back：
請說明：
1. 串列和字典的資料組織方式有什么不同？
2. 為什麼 departments[0] 是第一筆，不是第二筆？
3. sample_student["department"] 中的 "department" 是什麼角色？



In [7]:
# 建立科系串列
departments = ["資訊管理", "企業管理", "金融管理"]

# 顯示完整串列
print("完整科系串列：", departments)

# 取得第一個元素，索引從 0 開始
print("第一個科系：", departments[0])

# 計算串列中的元素數量
print("串列長度：", len(departments))

# 使用 for 迴圈依序印出每個科系
for department in departments:
    print("目前科系：", department)

# 建立一個學生字典
sample_student = {
    "student_id": 1,
    "department": "資訊管理",
    "sleep_hours": 7.0,
    "quiz_score": 82
}

# 用鍵取得字典中的值
print("學生科系：", sample_student["department"])
print("小考成績：", sample_student["quiz_score"])


完整科系串列： ['資訊管理', '企業管理', '金融管理']
第一個科系： 資訊管理
串列長度： 3
目前科系： 資訊管理
目前科系： 企業管理
目前科系： 金融管理
學生科系： 資訊管理
小考成績： 82


In [8]:
# 修改學生成績
sample_student["quiz_score"] = 76

# 檢查修改結果
print("修改後成績：", sample_student["quiz_score"])


修改後成績： 76


##Mission 4：算術運算與成績公式
#Knowledge Point


*   Python 可以用 +、-、* 進行算術運算。
*   計算公式可以拆成多個步驟。
*   先計算每一項影響，再合計，會比較容易理解。
*   本單元使用的是模擬公式，不表示真實生活中的因果關係。



公式：
小考成績
= 50
+ 3 × 讀書時數
+ 2 × 睡眠時數
- 1.2 × 螢幕使用時數
+ 0.15 × 出席率

執行前預測：
如果 study_hours 增加，模擬成績應該增加還是減少？

Try It：
把 study_hours 改成 4.0，或把 screen_time_hours 改成 6.0，重新執行。

Verification：
觀察成績是否隨你修改的數值改變。
注意：這只是模擬公式，不代表真實因果。

Explain-back：
請說明：
1. study_effect 代表什麼？
2. screen_effect 為什麼是負數？
3. 為什麼這個公式只能視為模擬，不能直接解釋為真實原因？


In [9]:
# 假想學生資料
study_hours = 2.5
sleep_hours = 7.0
screen_time_hours = 4.0
attendance_rate = 90

# 基礎分數
base_score = 50

# 分別計算每一項影響
study_effect = 3 * study_hours
sleep_effect = 2 * sleep_hours
screen_effect = -1.2 * screen_time_hours
attendance_effect = 0.15 * attendance_rate

# 合計模擬小考成績
quiz_score = base_score + study_effect + sleep_effect + screen_effect + attendance_effect

# 顯示計算過程
print("基礎分：", base_score)
print("讀書時數影響：", study_effect)
print("睡眠時數影響：", sleep_effect)
print("螢幕時間影響：", screen_effect)
print("出席率影響：", attendance_effect)
print("模擬小考成績：", quiz_score)


基礎分： 50
讀書時數影響： 7.5
睡眠時數影響： 14.0
螢幕時間影響： -4.8
出席率影響： 13.5
模擬小考成績： 80.2


In [10]:
# 修改讀書時數，觀察模擬成績改變
study_hours = 4.0
sleep_hours = 7.0
screen_time_hours = 4.0
attendance_rate = 90

base_score = 50

study_effect = 3 * study_hours
sleep_effect = 2 * sleep_hours
screen_effect = -1.2 * screen_time_hours
attendance_effect = 0.15 * attendance_rate

quiz_score = base_score + study_effect + sleep_effect + screen_effect + attendance_effect

print("修改讀書時數後的模擬小考成績：", quiz_score)


修改讀書時數後的模擬小考成績： 84.7


##Mission 5：條件判斷與自訂函式
#Knowledge Point


*   def 用來定義一個函式。

*   函式可以接收參數。
*   return 用來回傳結果。


*   if / elif / else 用來做條件判斷。

*   Python 使用縮排来表示程式區塊。
*   函式定義一次，之後可以重複呼叫。





執行前預測：
describe_score(85) 會回傳什麼？
describe_score(70) 會回傳什麼？
describe_score(55) 會回傳什麼？

Try It：
用 100、60、0 三個分數測試函式。

Verification：
檢查 80 分以上回傳「表現良好」。
60 到 79 分回傳「達到基本標準」。
60 分以下回傳「需要加強」。

Explain-back：
請說明：
1. describe_score 的參數 score 是什麼？
2. return 的作用是什麼？
3. 為什麼 if 下面的程式碼需要縮排？


In [11]:
# 定義一個用來描述成績的函式
def describe_score(score):
    # 如果 score 大於等於 80
    if score >= 80:
        return "表現良好"
    # 如果 score 大於等於 60
    elif score >= 60:
        return "達到基本標準"
    # 其他情況
    else:
        return "需要加強"

# 呼叫函式
print("85 分：", describe_score(85))
print("70 分：", describe_score(70))
print("55 分：", describe_score(55))


85 分： 表現良好
70 分： 達到基本標準
55 分： 需要加強


In [12]:
# 用不同分數測試函式
print("100 分：", describe_score(100))
print("60 分：", describe_score(60))
print("0 分：", describe_score(0))


100 分： 表現良好
60 分： 達到基本標準
0 分： 需要加強


##Mission 6：匯入資料科學套件
#Knowledge Point

*   套件是别人整理好的程式工具。
*   import 用來匯入套件。
*   numpy 常用於數值運算與亂數產生，縮寫常寫成 np。
*   pandas 常用於資料表操作，縮寫常寫成 pd。
*   函式、方法、屬性的差別：
- 函式：例如 print()
- 方法：例如 df.head()
- 屬性：例如 df.shape
*   使用相同的亂數種子，可以讓產生出的隨機資料可重現。



執行前預測：
df.shape 會顯示什麼？
df.head() 會顯示什麼？

Try It：
把 test_seed 改成 2026，再執行一次，觀察輸出的亂數是否改變。

Verification：
確認 import 之後可以使用 np 和 pd。
確認同一個 seed 會產生相同的隨機結果。

Explain-back：
請說明：
1. import numpy as np 表示什麼？
2. print() 為什麼可以視為函式？
3. df.head() 中的 head() 為什麼稱為方法？
4. df.shape 為什麼稱為屬性？


In [13]:
# 匯入 numpy 與 pandas
import numpy as np
import pandas as pd

# 建立可重現的亂數產生器
random_seed = 42
rng = np.random.default_rng(random_seed)

# 建立一個很小的 DataFrame 作為示範
demo_df = pd.DataFrame({
    "number": [10, 20, 30]
})

# 使用 print() 函式
print("這是 print() 函式。")

# 使用屬性 shape
print("demo_df 的形狀：", demo_df.shape)

# 使用方法 head()
print("demo_df 前五筆：")
print(demo_df.head())


這是 print() 函式。
demo_df 的形狀： (3, 1)
demo_df 前五筆：
   number
0      10
1      20
2      30


In [14]:
# 使用另一個亂數種子做比較
test_seed = 101
test_rng = np.random.default_rng(test_seed)

# 產生一個隨機數
print("使用種子", test_seed, "產生的一個數：", test_rng.normal())


使用種子 101 產生的一個數： -0.7901524999630146


##Mission 7：建立模擬的大學生學習資料
#Knowledge Point
* pandas.DataFrame 可以想成 Python 中的資料表。
* 一列通常代表一個觀測對象，這裡是一一位學生。
* 一欄代表一個變數，例如睡眠時數、讀書時數。
* df 是儲存 DataFrame 的變數名稱。
* 本單元使用常態分布來產生有差異的模擬資料
* 不需要先理解常態分布的數學原理，只要知道它可以產生有平均、有變異的資料。

執行前預測：
df.head() 應該顯示幾列？
df 應該有幾欄？

Try It：
把 random_seed 改成 123，重新執行本 Mission 的 Code Cell，觀察資料是否改變。
完成後請改回 42，讓老師和學生可以產生相同資料。

Verification：
1. df.shape 是否為 (60, 6)？
2. sleep_hours 是否約在 6.7 附近有變化？
3. screen_time_hours 是否被限制在 1 到 10？
4. study_hours 是否被限制在 0.5 到 6？
5. attendance_rate 是否被限制在 60 到 100？

Explain-back：
請說明：
1. 一列代表什麼？
2. 一欄代表什麼？
3. df 是什麼？
4. 為什麼要使用 random_seed？


In [15]:
# 固定學生人數為 60 人
student_count = 60
n = student_count

# 建立科系選項
departments = ["資訊管理", "企業管理", "金融管理"]

# 固定亂數種子，讓資料可重現
random_seed = 42
rng = np.random.default_rng(random_seed)

# 學生編號：1 到 60
student_id_values = list(range(1, n + 1))

# 隨機選擇科系
department_values = rng.choice(departments, size=n)

# 睡眠時數：平均 6.7，標準差 1.1
sleep_raw = rng.normal(loc=6.7, scale=1.1, size=n)
sleep_values = np.round(sleep_raw, 1)

# 螢幕使用時數：平均 5.0，標準差 1.5，限制在 1 到 10
screen_raw = rng.normal(loc=5.0, scale=1.5, size=n)
screen_clipped = np.clip(screen_raw, 1.0, 10.0)
screen_values = np.round(screen_clipped, 1)

# 讀書時數：平均 2.5，標準差 1.0，限制在 0.5 到 6
study_raw = rng.normal(loc=2.5, scale=1.0, size=n)
study_clipped = np.clip(study_raw, 0.5, 6.0)
study_values = np.round(study_clipped, 1)

# 出席率：平均 88，標準差 8，限制在 60 到 100
attendance_raw = rng.normal(loc=88, scale=8, size=n)
attendance_clipped = np.clip(attendance_raw, 60, 100)
attendance_values = np.round(attendance_clipped).astype(int)

# 建立 DataFrame
df = pd.DataFrame({
    "student_id": student_id_values,
    "department": department_values,
    "sleep_hours": sleep_values,
    "screen_time_hours": screen_values,
    "study_hours": study_values,
    "attendance_rate": attendance_values
})

# 顯示前五筆資料
df.head()


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate
0,1,資訊管理,9.1,6.0,2.3,95
1,2,金融管理,6.3,4.9,1.5,77
2,3,企業管理,6.1,4.4,0.8,88
3,4,企業管理,5.8,4.9,2.0,84
4,5,企業管理,7.4,2.5,2.4,85


In [16]:
# 檢查資料大小
print("資料大小：", df.shape)

# 檢查各欄位範圍
print("睡眠時數範圍：", df["sleep_hours"].min(), "到", df["sleep_hours"].max())
print("螢幕時間範圍：", df["screen_time_hours"].min(), "到", df["screen_time_hours"].max())
print("讀書時數範圍：", df["study_hours"].min(), "到", df["study_hours"].max())
print("出席率範圍：", df["attendance_rate"].min(), "到", df["attendance_rate"].max())


資料大小： (60, 6)
睡眠時數範圍： 4.8 到 9.1
螢幕時間範圍： 1.8 到 9.4
讀書時數範圍： 0.5 到 4.6
出席率範圍： 71 到 100


##Mission 8：建立小考成績欄位
#Knowledge Point
* 可以用 DataFrame 中的欄位進行計算。
* df["欄位名稱"] 可以選取一欄資料。
* 中括號內放多個欄位名稱，可以選取多欄。
* np.clip() 可以用來把數值限制在指定範圍。
* 本單元的成績公式仍然是模擬公式，不表示真實因果。

公式：
小考成績
= 50
+ 3 × 讀書時數
+ 2 × 睡眠時數
- 1.2 × 螢幕使用時數
+ 0.15 × 出席率
+ 平均數為 0、標準差為 5 的隨機誤差

執行前預測：
quiz_score 是否可能全部相同？
為什麼不會？

Try It：
把 quiz_base 改成 55，重新執行本 Mission 的 Code Cell。

Verification：
1. quiz_score 是否都在 0 到 100 之間？
2. 成績是否因為隨機誤差而不完全相同？
3. 選取多欄時，中括號裡放了什麼？

Explain-back：
請說明：
1. df["study_hours"] 代表什麼？
2. 為什麼要用 np.clip() 限制成績？
3. df[["student_id", "study_score", ...]] 的作用是什麼？


In [17]:
# 建立成績欄位時使用另一個固定亂數產生器，
# 這樣即使重複執行本單元，也可以得到相同的成績誤差。
score_seed = 42
score_rng = np.random.default_rng(score_seed)

# 基礎分數
quiz_base = 50

# 分別計算各項影響
study_effect = 3 * df["study_hours"]
sleep_effect = 2 * df["sleep_hours"]
screen_effect = -1.2 * df["screen_time_hours"]
attendance_effect = 0.15 * df["attendance_rate"]

# 隨機誤差：平均 0，標準差 5
quiz_error = score_rng.normal(loc=0, scale=5, size=len(df))

# 合計原始成績
quiz_raw = (
    quiz_base
    + study_effect
    + sleep_effect
    + screen_effect
    + attendance_effect
    + quiz_error
)

# 限制成績在 0 到 100 之間
quiz_clipped = np.clip(quiz_raw, 0, 100)

# 四捨五入成整數，並加入 DataFrame
df["quiz_score"] = np.round(quiz_clipped).astype(int)

# 選取多個欄位，顯示前五筆
df[
    [
        "student_id",
        "study_hours",
        "sleep_hours",
        "screen_time_hours",
        "attendance_rate",
        "quiz_score"
    ]
].head()


,student_id,study_hours,sleep_hours,screen_time_hours,attendance_rate,quiz_score
0,1,2.3,9.1,6.0,95,84
1,2,1.5,6.3,4.9,77,68
2,3,0.8,6.1,4.4,88,76
3,4,2.0,5.8,4.9,84,79
4,5,2.4,7.4,2.5,85,72


In [18]:
# 將基礎分數改成 55，再重新計算成績
score_seed = 42
score_rng = np.random.default_rng(score_seed)

quiz_base = 55

study_effect = 3 * df["study_hours"]
sleep_effect = 2 * df["sleep_hours"]
screen_effect = -1.2 * df["screen_time_hours"]
attendance_effect = 0.15 * df["attendance_rate"]

quiz_error = score_rng.normal(loc=0, scale=5, size=len(df))

quiz_raw = (
    quiz_base
    + study_effect
    + sleep_effect
    + screen_effect
    + attendance_effect
    + quiz_error
)

quiz_clipped = np.clip(quiz_raw, 0, 100)
df["quiz_score"] = np.round(quiz_clipped).astype(int)

print("基礎分數改成 55 後的前五筆成績：")
df[["student_id", "quiz_score"]].head()


基礎分數改成 55 後的前五筆成績：


,student_id,quiz_score
0,1,89
1,2,73
2,3,81
3,4,84
4,5,77


In [19]:
# 檢查成績範圍
print("quiz_score 最小值：", df["quiz_score"].min())
print("quiz_score 最大值：", df["quiz_score"].max())


quiz_score 最小值： 73
quiz_score 最大值： 95


##Mission 9：加入缺失值與異常值
#Knowledge Point
* 真實資料常常有缺失值。
* np.nan 代表「不是數值」，常用來表示缺失。
* .loc 可以用來根據索引選取或修改特定資料。
* DataFrame 的索引預設從 0 開始。
* 索引 5 對應 student_id 6。
* 索引 17 對應 student_id 18。
* 異常值是明顯不符合合理範圍的數值。


本單元會故意加入三筆資料品質問題：
1. 索引 5 的 sleep_hours 缺失
2. 索引 17 的 sleep_hours 缺失
3. 索引 8 的 screen_time_hours 異常，變成 15.0 小時

執行前預測：
加入缺失值後，df["sleep_hours"] 會有幾筆缺失？

Try It：
嘗試執行 df.loc[[8]]，觀察異常值那一列。

Verification：
1. 是否找到 student_id 6？
2. 是否找到 student_id 18？
3. 是否找到 screen_time_hours = 15.0 的學生？

Explain-back：
請說明：
1. .loc 是做什麼用的？
2. 為什麼索引 5 對應 student_id 6？
3. 15.0 小時的螢幕使用時間為什麼算異常值？


In [20]:
# 將索引 5 與索引 17 的 sleep_hours 設為缺失值
df.loc[[5, 17], "sleep_hours"] = np.nan

# 將索引 8 的 screen_time_hours 設為異常值
df.loc[8, "screen_time_hours"] = 15.0

# 找出這三筆問題資料
df.loc[[5, 8, 17]]


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
5,6,金融管理,NaN,2.8,4.3,96,88
8,9,資訊管理,5.8,15.0,2.0,87,79
17,18,資訊管理,NaN,4.5,2.9,100,82


In [21]:
# 單獨查看索引 8 的資料
df.loc[[8]]


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
8,9,資訊管理,5.8,15.0,2.0,87,79


In [23]:
# 檢查 sleep_hours 的缺失值數量
print("sleep_hours 缺失值數量：", df["sleep_hours"].isna().sum())

# 檢查是否有超過 10 小時的螢幕使用時間
high_screen_count = (df["screen_time_hours"] > 10).sum()
print("螢幕時間超過 10 小時的數量：", high_screen_count)


sleep_hours 缺失值數量： 2
螢幕時間超過 10 小時的數量： 1


##Mission 10：第一次資料品質檢查
#Knowledge Point
* df.head()：顯示前幾列。
* df.shape：顯示資料表的列數與欄數。
* df.columns：顯示所有欄位名稱。
* df.dtypes：顯示每一欄的資料型態。
* df.isna().sum()：計算每一欄的缺失值數量。
* df.describe()：顯示常見描述統計。

執行前預測：
請先猜測 df.shape 會顯示什麼？
df.isna().sum() 會顯示哪一欄有缺失？

Try It：
執行 df.tail()，看看資料最後幾列。

Verification：
請回答：
1. 資料共有幾列、幾欄？
2. 哪個欄位有缺失值？
3. 缺失值共有幾筆？
4. 哪個欄位可能存在異常值？
5. 如何從描述統計發現該異常值？

Explain-back：
請選擇 df.describe() 中的一個欄位，說明 min、mean、max 各代表什麼。


In [24]:
# 顯示前五筆資料
df.head()


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
0,1,資訊管理,9.1,6.0,2.3,95,89
1,2,金融管理,6.3,4.9,1.5,77,73
2,3,企業管理,6.1,4.4,0.8,88,81
3,4,企業管理,5.8,4.9,2.0,84,84
4,5,企業管理,7.4,2.5,2.4,85,77


In [25]:
# 顯示資料大小：(列數, 欄數)
df.shape


(60, 7)

In [26]:
# 顯示所有欄位名稱
df.columns


Index(['student_id', 'department', 'sleep_hours', 'screen_time_hours',
       'study_hours', 'attendance_rate', 'quiz_score'],
      dtype='object')

In [27]:
# 顯示每一欄的資料型態
df.dtypes


,0
student_id,int64
department,object
sleep_hours,float64
screen_time_hours,float64
study_hours,float64
attendance_rate,int64
quiz_score,int64


In [28]:
# 計算每一欄的缺失值數量
df.isna().sum()


,0
student_id,0
department,0
sleep_hours,2
screen_time_hours,0
study_hours,0
attendance_rate,0
quiz_score,0


In [29]:
# 顯示描述統計
df.describe()


,student_id,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
count,60.000000,58.000000,60.000000,60.000000,60.000000,60.000000
mean,30.500000,6.682759,4.946667,2.473333,87.166667,83.450000
std,17.464249,0.822307,1.985239,0.965659,7.300259,5.369168
min,1.000000,4.800000,1.800000,0.500000,71.000000,73.000000
25%,15.750000,6.200000,3.600000,1.700000,82.000000,79.000000
50%,30.500000,6.750000,4.700000,2.400000,87.000000,84.000000
75%,45.250000,7.300000,5.625000,2.950000,91.000000,88.000000
max,60.000000,9.100000,15.000000,4.600000,100.000000,95.000000


In [30]:
# 顯示最後五筆資料
df.tail()


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
55,56,資訊管理,7.2,4.4,2.5,96,83
56,57,企業管理,6.2,4.1,1.7,89,80
57,58,資訊管理,6.7,4.8,1.3,91,88
58,59,金融管理,7.2,6.6,1.6,88,75
59,60,金融管理,7.2,5.2,2.2,88,88


##Mission 11：篩選與比較資料
#Knowledge Point
* df["欄位"]：選取單一欄位。
* df[["欄位A", "欄位B"]]：選取多個欄位。
* 布林條件可以產生 True 或 False。
* 布林條件可以放在 df[條件] 中，用來篩選列。
* isna() 可以用來找出缺失值。

執行前預測：
df["quiz_score"] >= 80 會產生什麼？
df[high_score_condition] 會顯示什麼？

Try It：
把成績條件從 80 分改成 90 分，觀察篩選出的學生數量是否改變。

Verification：
1. 有缺失值的資料是否正確顯示？
2. 螢幕時間超過 10 小時的資料是否只有異常值那筆？
3. 80 分以上的學生是否符合條件？

Explain-back：
請說明：
1. high_score_condition 裡面存放的是什麼？
2. df[high_score_condition] 是如何篩選資料的？
3. 為什麼條件中要用 >= 而不是 =？


In [31]:
# 選取單一欄位
one_column = df["department"]

# 顯示前五筆
one_column.head()


,department
0,資訊管理
1,金融管理
2,企業管理
3,企業管理
4,企業管理


In [32]:
# 選取多個欄位
selected_columns = df[["student_id", "department", "quiz_score"]]

# 顯示前五筆
selected_columns.head()


,student_id,department,quiz_score
0,1,資訊管理,89
1,2,金融管理,73
2,3,企業管理,81
3,4,企業管理,84
4,5,企業管理,77


In [33]:
# 找出 sleep_hours 有缺失值的資料
missing_sleep_rows = df[df["sleep_hours"].isna()]

# 顯示結果
missing_sleep_rows


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
5,6,金融管理,NaN,2.8,4.3,96,88
17,18,資訊管理,NaN,4.5,2.9,100,82


In [34]:
# 建立布林條件
high_screen_condition = df["screen_time_hours"] > 10

# 使用條件篩選資料
high_screen_rows = df[high_screen_condition]

# 顯示結果
high_screen_rows


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
8,9,資訊管理,5.8,15.0,2.0,87,79


In [35]:
# 建立布林條件
high_score_condition = df["quiz_score"] >= 80

# 使用條件篩選資料
high_score_students = df[high_score_condition]

# 顯示結果
high_score_students


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
0,1,資訊管理,9.1,6.0,2.3,95,89
2,3,企業管理,6.1,4.4,0.8,88,81
3,4,企業管理,5.8,4.9,2.0,84,84
5,6,金融管理,NaN,2.8,4.3,96,88
6,7,資訊管理,6.6,3.0,2.6,92,87
7,8,金融管理,5.8,3.5,3.5,99,86
10,11,企業管理,7.5,4.4,1.5,86,87
11,12,金融管理,7.3,6.9,1.8,90,84
12,13,金融管理,6.0,4.5,4.6,89,89
13,14,金融管理,7.0,6.1,1.7,79,84


In [36]:
# 改成 90 分以上的學生
new_score_condition = df["quiz_score"] >= 90
new_score_students = df[new_score_condition]

print("90 分以上的學生數量：", len(new_score_students))
new_score_students[["student_id", "quiz_score"]]


90 分以上的學生數量： 6


,student_id,quiz_score
14,15,90
16,17,93
30,31,95
35,36,93
39,40,91
51,52,91


##Mission 12：儲存 CSV 檔案
#Knowledge Point
* CSV 是一種純文字表格檔，常用於儲存資料。
* df.to_csv() 可以把 DataFrame 寫入 CSV 檔案。
* index=False 表示不把左邊的索引欄儲存進去。
* encoding="utf-8-sig" 有助於中文在 Excel 中正常顯示。
* Notebook 中的變數存在記憶體中；檔案是寫到外部儲存空間。
* Google Colab 可以使用 files.download() 下載檔案。

執行前預測：
如果 index=True，儲存時會多一欄嗎？

Try It：
把 file_name 改成 "my_student_data.csv"，再執行儲存。

Verification：
1. 是否看到儲存成功的訊息？
2. 如果是在 Google Colab，是否出現下載選項或下載進度？
3. 如果不在 Colab，檔案會存在 Notebook 的工作目錄中。

Explain-back：
請說明：
1. CSV 是什麼？
2. index=False 的作用是什麼？
3. 為什麼中文資料可以使用 utf-8-sig？
4. Notebook 中的 df 和 CSV 檔案有什么不同？


In [37]:
# 設定檔案名稱
file_name = "student_learning_data.csv"

# 將 DataFrame 儲存為 CSV
df.to_csv(
    file_name,
    index=False,
    encoding="utf-8-sig"
)

# 顯示儲存訊息
print("資料已儲存為：", file_name)


資料已儲存為： student_learning_data.csv


In [38]:
# 改用另一個檔案名稱
file_name = "my_student_data.csv"

df.to_csv(
    file_name,
    index=False,
    encoding="utf-8-sig"
)

print("新檔案已儲存為：", file_name)


新檔案已儲存為： my_student_data.csv


In [39]:
# 這只是提醒：檔案與記憶體變數不同。
# df 是 Notebook 記憶體中的資料。
# CSV 是寫到外部系統的文字檔案。
print("檔案與記憶體變數的不同：")
print("df 存在 Notebook 的記憶體中。")
print("CSV 是寫到外部儲存空間的檔案。")


檔案與記憶體變數的不同：
df 存在 Notebook 的記憶體中。
CSV 是寫到外部儲存空間的檔案。


In [40]:
# 這段下載程式只適用於 Google Colab。
# 如果不是 Colab 環境，會顯示提醒訊息。
try:
    from google.colab import files
    files.download(file_name)
except Exception:
    print("目前不是 Google Colab 環境。請在 Notebook 工作目錄中開啟 CSV 檔案。")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##最終整合任務：完整可重新執行程式
#Knowledge Point
* 整合版程式會從套件匯入開始。
* 建立 60 筆模擬資料。
* 建立 quiz_score。
* 加入兩筆睡眠缺失值。
* 加入一筆螢幕時間異常值。
* 顯示前五筆資料。
* 檢查資料大小、資料型態、缺失值與描述統計。
* 匯出 CSV 檔案。
* 所有欄位名稱與前面保持一致。

執行前預測：
這個整合版程式執行後，df 應該有幾列？
應該有幾個欄位？
應該有幾筆 sleep_hours 缺失？

Try It：
把 random_seed 改成 123，重新執行，觀察資料是否改變。

Verification：
1. df.shape 是否為 (60, 7)？
2. sleep_hours 是否有 2 筆缺失？
3. 是否出現 screen_time_hours = 15.0？
4. CSV 是否成功儲存？

Explain-back：
請從整合版程式中挑出三個你覺得重要的區塊，說明它們各自動工是什麼。


In [41]:
# 整合版：建立模擬資料、計算成績、加入問題、檢查並匯出 CSV
import numpy as np
import pandas as pd

# 基本設定
student_count = 60
n = student_count
departments = ["資訊管理", "企業管理", "金融管理"]
random_seed = 42
rng = np.random.default_rng(random_seed)

# 建立學生編號
student_id_values = list(range(1, n + 1))

# 隨機選擇科系
department_values = rng.choice(departments, size=n)

# 建立睡眠時數
sleep_raw = rng.normal(loc=6.7, scale=1.1, size=n)
sleep_values = np.round(sleep_raw, 1)

# 建立螢幕使用時數
screen_raw = rng.normal(loc=5.0, scale=1.5, size=n)
screen_clipped = np.clip(screen_raw, 1.0, 10.0)
screen_values = np.round(screen_clipped, 1)

# 建立讀書時數
study_raw = rng.normal(loc=2.5, scale=1.0, size=n)
study_clipped = np.clip(study_raw, 0.5, 6.0)
study_values = np.round(study_clipped, 1)

# 建立出席率
attendance_raw = rng.normal(loc=88, scale=8, size=n)
attendance_clipped = np.clip(attendance_raw, 60, 100)
attendance_values = np.round(attendance_clipped).astype(int)

# 建立 DataFrame
df = pd.DataFrame({
    "student_id": student_id_values,
    "department": department_values,
    "sleep_hours": sleep_values,
    "screen_time_hours": screen_values,
    "study_hours": study_values,
    "attendance_rate": attendance_values
})

# 建立小考成績
score_seed = 42
score_rng = np.random.default_rng(score_seed)

quiz_base = 50

study_effect = 3 * df["study_hours"]
sleep_effect = 2 * df["sleep_hours"]
screen_effect = -1.2 * df["screen_time_hours"]
attendance_effect = 0.15 * df["attendance_rate"]

quiz_error = score_rng.normal(loc=0, scale=5, size=len(df))

quiz_raw = (
    quiz_base
    + study_effect
    + sleep_effect
    + screen_effect
    + attendance_effect
    + quiz_error
)

quiz_clipped = np.clip(quiz_raw, 0, 100)
df["quiz_score"] = np.round(quiz_clipped).astype(int)

# 加入缺失值與異常值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0

# 顯示前五筆資料
print("前五筆資料：")
print(df.head())

# 檢查資料大小
print("資料大小：", df.shape)

# 檢查資料型態
print("資料型態：")
print(df.dtypes)

# 檢查缺失值
print("缺失值統計：")
print(df.isna().sum())

# 描述統計
print("描述統計：")
print(df.describe())

# 匯出 CSV
file_name = "student_learning_data.csv"

df.to_csv(
    file_name,
    index=False,
    encoding="utf-8-sig"
)

print("資料已儲存為：", file_name)

# Google Colab 下載；非 Colab 環境會顯示提醒
try:
    from google.colab import files
    files.download(file_name)
except Exception:
    print("若非 Google Colab 環境，請在工作目錄中開啟 CSV 檔案。")


前五筆資料：
   student_id department  sleep_hours  screen_time_hours  study_hours  \
0           1       資訊管理          9.1                6.0          2.3   
1           2       金融管理          6.3                4.9          1.5   
2           3       企業管理          6.1                4.4          0.8   
3           4       企業管理          5.8                4.9          2.0   
4           5       企業管理          7.4                2.5          2.4   

   attendance_rate  quiz_score  
0               95          84  
1               77          68  
2               88          76  
3               84          79  
4               85          72  
資料大小： (60, 7)
資料型態：
student_id             int64
department            object
sleep_hours          float64
screen_time_hours    float64
study_hours          float64
attendance_rate        int64
quiz_score             int64
dtype: object
缺失值統計：
student_id           0
department           0
sleep_hours          2
screen_time_hours    0
study_hours         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##最終學習檢核


請不使用選擇題，用自己的話回答以下五題：

1. 變數與資料值有什麼不同？
2. 串列與字典有什麼不同？
3. 函式與方法有什麼不同？
4. 為什麼程式能執行不代表分析結果一定正確？
5. 為什麼模擬資料中的相關關係不能直接解釋為真實因果關係？

Explain-back 任務：
請選擇三行你原本看不懂的程式碼，分別說明：
1. 它的輸入是什麼？
2. 它做了什麼處理？
3. 它的輸出是什麼？
4. 你如何驗證執行結果？

提醒：
本教材使用模擬資料建立學習資料表。
模擬資料可以幫助學習 Python、pandas 與基本資料檢查，
但不能用來推論真實學生行為，也不能證明讀書、睡眠、螢幕時間與成績之間的真實因果關係。
